## 05c — ML Model: Single-Bidder Classifier

Trains and evaluates three binary classifiers to predict whether a public procurement tender will attract only one bidder. XGBoost is selected as the final model.

**Input:** `workspace.gold.ml_features_train` / `workspace.gold.ml_features_test`  
**Output:** `/Volumes/workspace/default/lakehouse/ml/bidders_xgb_final.pkl`

Models compared: logistic regression (interpretable baseline), random forest, XGBoost. All three use an identical sklearn Pipeline with OHE for categoricals and StandardScaler for numeric features. Primary metric is ROC-AUC. The notebook also includes a no-value variant of each model to evaluate the cost of dropping award value for CN/CAN parity reasons — the delta is at most 0.007 AUC.

In [0]:
%pip install xgboost

In [0]:
# Goal: load the feature tables into pandas and confirm shapes and null counts
# before building the pipeline.
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import classification_report, roc_auc_score, RocCurveDisplay
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

CAT_FEATURES = ["country_clean", "cpv_division_clean", "procedure_clean", "buyer_type_clean"]
NUM_FEATURES  = ["log_award_value", "value_missing"]
TARGET        = "is_single_bidder"

train_pd = spark.table("workspace.gold.ml_features_train").toPandas()
test_pd  = spark.table("workspace.gold.ml_features_test").toPandas()

print("Train:", train_pd.shape, "| Test:", test_pd.shape)
print("\nNull counts in train:")
print(train_pd.isnull().sum())

In [0]:
# Conclusion from previous: data loaded cleanly, zero nulls across all features.
# Goal: build a sklearn Pipeline for each of the three models. The preprocessor handles
# encoding and scaling in one step, ensuring identical transformations on train and test.

X_train = train_pd[CAT_FEATURES + NUM_FEATURES]
y_train = train_pd[TARGET]
X_test  = test_pd[CAT_FEATURES + NUM_FEATURES]
y_test  = test_pd[TARGET]

preprocessor = ColumnTransformer(transformers=[
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_FEATURES),
    ("num", StandardScaler(), NUM_FEATURES),
])

lr_pipeline = Pipeline([
    ("prep", preprocessor),
    ("clf",  LogisticRegression(max_iter=1000, random_state=42)),
])

rf_pipeline = Pipeline([
    ("prep", preprocessor),
    ("clf",  RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
])

xgb_pipeline = Pipeline([
    ("prep", preprocessor),
    ("clf",  XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                           random_state=42, eval_metric="logloss", verbosity=0)),
])

print("Pipelines defined.")
print("X_train shape:", X_train.shape)

In [0]:
# Conclusion from previous: pipelines defined for all three models with identical
# preprocessor, ensuring results are directly comparable.
# Goal: train logistic regression, random forest, and XGBoost on the training set.
import time

for name, pipeline in [
    ("Logistic Regression", lr_pipeline),
    ("Random Forest",       rf_pipeline),
    ("XGBoost",             xgb_pipeline),
]:
    t0 = time.time()
    pipeline.fit(X_train, y_train)
    print(f"{name} trained in {time.time()-t0:.1f}s")

In [0]:
# Conclusion from previous: all three models trained successfully.
# Goal: evaluate all three on the held-out test set. Primary metric is ROC-AUC
# (threshold-independent); classification report gives precision/recall context.

for name, pipeline in [
    ("Logistic Regression", lr_pipeline),
    ("Random Forest",       rf_pipeline),
    ("XGBoost",             xgb_pipeline),
]:
    y_pred  = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    auc     = roc_auc_score(y_test, y_proba)

    print(f"{'='*50}")
    print(f"  {name}  |  ROC-AUC: {auc:.4f}")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred, target_names=["competitive", "single-bidder"]))

In [0]:
# Conclusion from previous: XGBoost achieves the highest AUC and is selected as
# the primary model.
# Goal: plot ROC curves for all three models and XGBoost feature importances grouped
# by original variable. OHE splits categorical importance across many columns so
# grouping is necessary for a fair comparison.

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor("#F8F9FA")
for ax in axes:
    ax.set_facecolor("#F8F9FA")
    ax.spines[["top", "right"]].set_visible(False)

# Left: ROC curves
for name, pipeline, color in [
    ("Logistic Regression", lr_pipeline,  "#888780"),
    ("Random Forest",       rf_pipeline,  "#E87722"),
    ("XGBoost",             xgb_pipeline, "#1A6EBD"),
]:
    RocCurveDisplay.from_estimator(pipeline, X_test, y_test, name=name, color=color, ax=axes[0])

axes[0].plot([0,1],[0,1], "k--", linewidth=0.8, label="Random (AUC=0.500)")
axes[0].set_title("ROC Curve", fontsize=13, fontweight="bold")
axes[0].legend(fontsize=9)

# Right: XGBoost importances grouped by original variable
ohe_features = list(
    xgb_pipeline.named_steps["prep"]
    .named_transformers_["cat"]
    .get_feature_names_out(CAT_FEATURES)
)
all_features = ohe_features + NUM_FEATURES
importances = pd.Series(xgb_pipeline.named_steps["clf"].feature_importances_, index=all_features)

def get_group(col):
    for feat in CAT_FEATURES:
        if col.startswith(feat):
            return feat
    return col

grouped = importances.groupby(importances.index.map(get_group)).sum().sort_values()
grouped.plot(kind="barh", ax=axes[1], color="#1A6EBD", edgecolor="none")
axes[1].set_title("Feature importance by original variable (XGBoost)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Importance (sum across OHE columns)", fontsize=11)

plt.suptitle("Model evaluation — Single-Bidder Classifier", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("/tmp/model_evaluation.png", dpi=150, bbox_inches="tight")
plt.show()

In [0]:
# Conclusion from previous: XGBoost leads across all metrics. Feature importance
# grouped by variable shows which structural features drive predictions.
# Goal: retrain all three models without log_award_value and value_missing.
# Rationale: the model trains on award_value but inference uses estimated_value.
# The spread between both is variable and unmeasurable without the CN-CAN join,
# so including value introduces systematic noise rather than clean signal.

CAT_ONLY = CAT_FEATURES

preprocessor_nv = ColumnTransformer(transformers=[
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_ONLY),
])

lr_nv  = Pipeline([("prep", preprocessor_nv), ("clf", LogisticRegression(max_iter=1000, random_state=42))])
rf_nv  = Pipeline([("prep", preprocessor_nv), ("clf", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))])
xgb_nv = Pipeline([("prep", preprocessor_nv), ("clf", XGBClassifier(n_estimators=200, max_depth=6,
                    learning_rate=0.1, random_state=42, eval_metric="logloss", verbosity=0))])

for name, pipeline in [("Logistic Regression", lr_nv), ("Random Forest", rf_nv), ("XGBoost", xgb_nv)]:
    pipeline.fit(train_pd[CAT_ONLY], y_train)

auc_lr_wv  = roc_auc_score(y_test, lr_pipeline.predict_proba(X_test)[:,1])
auc_rf_wv  = roc_auc_score(y_test, rf_pipeline.predict_proba(X_test)[:,1])
auc_xgb_wv = roc_auc_score(y_test, xgb_pipeline.predict_proba(X_test)[:,1])

auc_lr_nv  = roc_auc_score(y_test, lr_nv.predict_proba(test_pd[CAT_ONLY])[:,1])
auc_rf_nv  = roc_auc_score(y_test, rf_nv.predict_proba(test_pd[CAT_ONLY])[:,1])
auc_xgb_nv = roc_auc_score(y_test, xgb_nv.predict_proba(test_pd[CAT_ONLY])[:,1])

print(f"{'Model':<25} {'With value':>12} {'Without value':>15} {'Delta':>8}")
print("-" * 62)
print(f"{'Logistic Regression':<25} {auc_lr_wv:>12.4f} {auc_lr_nv:>15.4f} {auc_lr_nv-auc_lr_wv:>+8.4f}")
print(f"{'Random Forest':<25} {auc_rf_wv:>12.4f} {auc_rf_nv:>15.4f} {auc_rf_nv-auc_rf_wv:>+8.4f}")
print(f"{'XGBoost':<25} {auc_xgb_wv:>12.4f} {auc_xgb_nv:>15.4f} {auc_xgb_nv-auc_xgb_wv:>+8.4f}")

In [0]:
# Conclusion from previous: removing award value costs at most 0.007 AUC across all
# models. Given the CN/CAN parity problem, the no-value version is the correct choice
# for production. XGBoost without value is the final model.
# Goal: persist the final model to disk for inference without retraining.
import joblib
import os

model_path = "/Volumes/workspace/default/lakehouse/ml/bidders_xgb_final.pkl"
os.makedirs(os.path.dirname(model_path), exist_ok=True)
joblib.dump(xgb_nv, model_path)

print(f"Model saved: {model_path}")
print(f"Size: {os.path.getsize(model_path) / 1024:.1f} KB")
print(f"\nFeatures at inference time: {CAT_FEATURES}")